In [6]:
import heapq
import math
import time


# -------------------------------------------------
# LOAD PUZZLES FROM TEXT FILE
# -------------------------------------------------
def load_puzzles(filename):
    puzzles = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith("#"):
                continue

            puzzle = tuple(int(x) for x in line.split())
            puzzles.append(puzzle)

    return puzzles


# -------------------------------------------------
# BASIC HELPERS
# -------------------------------------------------
def get_size(state):
    size = int(math.sqrt(len(state)))
    if size * size != len(state):
        raise ValueError("Puzzle length must be a perfect square.")
    return size


def goal_state(size):
    return tuple(list(range(1, size * size)) + [0])


def print_board(state):
    size = get_size(state)
    for r in range(size):
        row = state[r * size:(r + 1) * size]
        print(" ".join("_" if x == 0 else str(x) for x in row))
    print()


# -------------------------------------------------
# SOLVABILITY CHECK
# -------------------------------------------------
def count_inversions(state):
    arr = [x for x in state if x != 0]
    inversions = 0

    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inversions += 1

    return inversions


def is_solvable(state):
    size = get_size(state)
    inversions = count_inversions(state)

    if size % 2 == 1:
        return inversions % 2 == 0

    blank_index = state.index(0)
    blank_row_from_top = blank_index // size
    blank_row_from_bottom = size - blank_row_from_top

    if blank_row_from_bottom % 2 == 0:
        return inversions % 2 == 1
    else:
        return inversions % 2 == 0


# -------------------------------------------------
# NEIGHBORS
# -------------------------------------------------
def get_neighbors(state):
    size = get_size(state)
    zero_index = state.index(0)
    row, col = divmod(zero_index, size)

    neighbors = []

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in directions:
        new_row = row + dr
        new_col = col + dc

        if 0 <= new_row < size and 0 <= new_col < size:
            new_zero_index = new_row * size + new_col
            new_state = list(state)

            new_state[zero_index], new_state[new_zero_index] = (
                new_state[new_zero_index],
                new_state[zero_index],
            )

            neighbors.append(tuple(new_state))

    return neighbors


# -------------------------------------------------
# HEURISTICS
# -------------------------------------------------
def misplaced_tiles(state, goal):
    count = 0
    for i in range(len(state)):
        if state[i] != 0 and state[i] != goal[i]:
            count += 1
    return count


def manhattan_distance(state, goal):
    size = get_size(state)
    total = 0

    goal_positions = {}
    for i, tile in enumerate(goal):
        goal_positions[tile] = divmod(i, size)

    for i, tile in enumerate(state):
        if tile == 0:
            continue

        current_row, current_col = divmod(i, size)
        goal_row, goal_col = goal_positions[tile]

        total += abs(current_row - goal_row) + abs(current_col - goal_col)

    return total


def linear_conflict(state, goal):
    size = get_size(state)
    manhattan = manhattan_distance(state, goal)

    goal_positions = {}
    for i, tile in enumerate(goal):
        goal_positions[tile] = divmod(i, size)

    conflicts = 0

    # Row conflicts
    for row in range(size):
        current_row_tiles = state[row * size:(row + 1) * size]
        row_tiles = []

        for col, tile in enumerate(current_row_tiles):
            if tile == 0:
                continue

            goal_row, goal_col = goal_positions[tile]
            if goal_row == row:
                row_tiles.append((col, goal_col))

        for i in range(len(row_tiles)):
            for j in range(i + 1, len(row_tiles)):
                if row_tiles[i][1] > row_tiles[j][1]:
                    conflicts += 1

    # Column conflicts
    for col in range(size):
        col_tiles = []
        for row in range(size):
            tile = state[row * size + col]
            if tile == 0:
                continue

            goal_row, goal_col = goal_positions[tile]
            if goal_col == col:
                col_tiles.append((row, goal_row))

        for i in range(len(col_tiles)):
            for j in range(i + 1, len(col_tiles)):
                if col_tiles[i][1] > col_tiles[j][1]:
                    conflicts += 1

    return manhattan + 2 * conflicts


# -------------------------------------------------
# PATH RECONSTRUCTION
# -------------------------------------------------
def reconstruct_path(came_from, current):
    path = [current]

    while current in came_from:
        current = came_from[current]
        path.append(current)

    path.reverse()
    return path


# -------------------------------------------------
# A* ALGORITHM
# -------------------------------------------------
def astar(start, heuristic_function, time_limit=30):
    size = get_size(start)
    goal = goal_state(size)

    if start == goal:
        return {
            "solved": True,
            "moves": 0,
            "expanded": 0,
            "generated": 1,
            "max_frontier": 1,
            "runtime": 0.0,
            "path": [start],
        }

    if not is_solvable(start):
        return {
            "solved": False,
            "moves": -1,
            "expanded": 0,
            "generated": 0,
            "max_frontier": 0,
            "runtime": 0.0,
            "path": [],
        }

    start_time = time.perf_counter()

    open_heap = []
    counter = 0

    g_score = {start: 0}
    came_from = {}
    closed_set = set()

    start_h = heuristic_function(start, goal)
    heapq.heappush(open_heap, (start_h, counter, start))

    expanded = 0
    generated = 1
    max_frontier = 1

    while open_heap:
        current_time = time.perf_counter()
        if current_time - start_time > time_limit:
            return {
                "solved": False,
                "moves": -1,
                "expanded": expanded,
                "generated": generated,
                "max_frontier": max_frontier,
                "runtime": current_time - start_time,
                "path": [],
            }

        _, _, current = heapq.heappop(open_heap)

        if current in closed_set:
            continue

        if current == goal:
            path = reconstruct_path(came_from, current)
            return {
                "solved": True,
                "moves": len(path) - 1,
                "expanded": expanded,
                "generated": generated,
                "max_frontier": max_frontier,
                "runtime": time.perf_counter() - start_time,
                "path": path,
            }

        closed_set.add(current)
        expanded += 1

        for neighbor in get_neighbors(current):
            if neighbor in closed_set:
                continue

            tentative_g = g_score[current] + 1

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                counter += 1
                f_score = tentative_g + heuristic_function(neighbor, goal)
                heapq.heappush(open_heap, (f_score, counter, neighbor))
                generated += 1

        if len(open_heap) > max_frontier:
            max_frontier = len(open_heap)

    return {
        "solved": False,
        "moves": -1,
        "expanded": expanded,
        "generated": generated,
        "max_frontier": max_frontier,
        "runtime": time.perf_counter() - start_time,
        "path": [],
    }


# -------------------------------------------------
# RUN EXPERIMENTS
# -------------------------------------------------
def run_all_puzzles(filename):
    puzzles = load_puzzles(filename)

    heuristics = {
        "Misplaced Tiles": misplaced_tiles,
        "Manhattan Distance": manhattan_distance,
        "Linear Conflict": linear_conflict,
    }

    print("=" * 90)
    print("N-PUZZLE A* SEARCH PROJECT")
    print("=" * 90)
    print()

    for puzzle_index, puzzle in enumerate(puzzles, start=1):
        print(f"PUZZLE {puzzle_index}")
        print("-" * 40)
        print("Initial State:")
        print_board(puzzle)

        for heuristic_name, heuristic_function in heuristics.items():
            print(f"Heuristic: {heuristic_name}")

            result = astar(puzzle, heuristic_function)

            print(f"  Solved       : {result['solved']}")
            print(f"  Moves        : {result['moves']}")
            print(f"  Expanded     : {result['expanded']}")
            print(f"  Generated    : {result['generated']}")
            print(f"  Max Frontier : {result['max_frontier']}")
            print(f"  Runtime (s)  : {result['runtime']:.6f}")
            print()

        print("=" * 90)
        print()


# -------------------------------------------------
# SUMMARY TABLE
# -------------------------------------------------
def run_summary_table(filename):
    puzzles = load_puzzles(filename)

    heuristics = {
        "Misplaced Tiles": misplaced_tiles,
        "Manhattan Distance": manhattan_distance,
        "Linear Conflict": linear_conflict,
    }

    print("\nSUMMARY TABLE")
    print("=" * 120)
    print(
        f"{'Puzzle':<8} {'Heuristic':<22} {'Solved':<8} {'Moves':<8} "
        f"{'Expanded':<10} {'Generated':<10} {'Frontier':<10} {'Runtime(s)':<12}"
    )
    print("-" * 120)

    for puzzle_index, puzzle in enumerate(puzzles, start=1):
        for heuristic_name, heuristic_function in heuristics.items():
            result = astar(puzzle, heuristic_function)

            print(
                f"{puzzle_index:<8} "
                f"{heuristic_name:<22} "
                f"{str(result['solved']):<8} "
                f"{result['moves']:<8} "
                f"{result['expanded']:<10} "
                f"{result['generated']:<10} "
                f"{result['max_frontier']:<10} "
                f"{result['runtime']:<12.6f}"
            )

    print("=" * 120)


# -------------------------------------------------
# MAIN
# -------------------------------------------------
if __name__ == "__main__":
    filename = "puzzle.txt"

    run_all_puzzles(filename)
    run_summary_table(filename)

N-PUZZLE A* SEARCH PROJECT

PUZZLE 1
----------------------------------------
Initial State:
7 2 4
5 _ 6
8 3 1

Heuristic: Misplaced Tiles
  Solved       : True
  Moves        : 20
  Expanded     : 3666
  Generated    : 5759
  Max Frontier : 2091
  Runtime (s)  : 0.037634

Heuristic: Manhattan Distance
  Solved       : True
  Moves        : 20
  Expanded     : 282
  Generated    : 452
  Max Frontier : 170
  Runtime (s)  : 0.004535

Heuristic: Linear Conflict
  Solved       : True
  Moves        : 20
  Expanded     : 165
  Generated    : 271
  Max Frontier : 106
  Runtime (s)  : 0.008130



SUMMARY TABLE
Puzzle   Heuristic              Solved   Moves    Expanded   Generated  Frontier   Runtime(s)  
------------------------------------------------------------------------------------------------------------------------
1        Misplaced Tiles        True     20       3666       5759       2091       0.037901    
1        Manhattan Distance     True     20       282        452        170 